In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns

cmap = sns.color_palette("vlag", as_cmap=True)

In [ ]:
table_s2_dir = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/CombinedModel_ss/statistic_analysis/Table_S2.xlsx"
table_s3_dir = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/CombinedModel_ss/statistic_analysis/Table_S3.xlsx"
save_dir_node = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/CombinedModel_ss/statistic_analysis/node_level_risk_heatmap.png"
save_dir_edge = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/CombinedModel_ss/statistic_analysis/edge_level_risk_heatmap.png"

In [ ]:
def p_to_star(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    return ""


def plot_corr_heatmap(
    rho_df,
    p_df,
    title,
    save_path,
    figsize=(7, 6),
    vmin=None,
    vmax=None,
    show_rho=True
):

    # symmetrical color scale around zero
    if vmin is None or vmax is None:
        max_abs = np.nanmax(np.abs(rho_df.values))
        vmin = -max_abs
        vmax = max_abs

    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

    fig, ax = plt.subplots(figsize=figsize)

    im = ax.imshow(rho_df.values, aspect="auto", cmap=cmap, norm=norm)

    # ticks
    ax.set_xticks(np.arange(rho_df.shape[1]))
    ax.set_xticklabels(rho_df.columns, rotation=45, ha="right")

    ax.set_yticks(np.arange(rho_df.shape[0]))
    ax.set_yticklabels(rho_df.index)

    pos = ax.get_position()
    ax.set_position([
        pos.x0,
        pos.y0,
        pos.width*0.8,
        pos.height*1.3
    ])
    ax.set_position([0.10, 0.18, 0.65, 0.70])

    # annotations
    for i in range(rho_df.shape[0]):
        for j in range(rho_df.shape[1]):
            rho = rho_df.iloc[i, j]
            p = p_df.iloc[i, j]

            if pd.isna(rho):
                continue

            star = p_to_star(p)

            if show_rho:
                text = f"{rho:.2f}{star}"
            else:
                text = star

            ax.text(j, i, text, ha="center", va="center", fontsize=9)

    # grid
    ax.grid(False)
    ax.set_xticks(np.arange(-0.5, rho_df.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, rho_df.shape[0], 1), minor=True)

    ax.grid(which="minor", color="white", linewidth=1)
    ax.tick_params(which="minor", bottom=False, left=False)

    ax.set_title(title, pad=12)

    cbar = fig.colorbar(
        im,
        ax=ax,
        fraction=0.046,
        pad=0.04
    )
    cbar.set_label("Spearman's ρ")

    plt.tight_layout()

    plt.savefig(save_path, dpi=600, bbox_inches="tight")

    plt.show()

In [ ]:
node_df = pd.read_excel(table_s2_dir)

node_df = node_df.rename(columns={
    "Unnamed: 0": "Feature_group",
    "Features": "Feature",
    "Spearman’s ρ": "rho",
    "FDR-adjusted P-value": "p_fdr"
})

node_group_order = [
    "Patch-level proportion",
    "Pixel-level proportion",
    "Relative tissue composition"
]

tissue_order = [
    "Tumor",
    "Necrosis",
    "Fibrosis",
    "Normal",
    "Inflammation",
    "Steatosis",
    "Bile duct reaction"
]


node_rho = node_df.pivot(
    index="Feature",
    columns="Feature_group",
    values="rho"
)

node_p = node_df.pivot(
    index="Feature",
    columns="Feature_group",
    values="p_fdr"
)

node_rho = node_rho.reindex(
    index=tissue_order,
    columns=node_group_order
)

node_p = node_p.reindex(
    index=tissue_order,
    columns=node_group_order
)


plot_corr_heatmap(
    rho_df=node_rho.T,
    p_df=node_p.T,
    title="Correlation between node-level tissue features and model-derived risk",
    save_path=save_dir_node,
    figsize=(8, 4),
    vmin=-0.35,
    vmax=0.35,
    show_rho=False
)

In [ ]:
edge_df = pd.read_excel(table_s3_dir)

edge_df = edge_df.rename(columns={
    "Unnamed: 0": "Feature_group",
    "Features": "Feature",
    "Spearman’s ρ": "rho",
    "FDR-adjusted P-value": "p_fdr"
})

edge_df["Feature_group"] = edge_df["Feature_group"].replace({
    "Importance-weighted edge proportion": "Importance-weighted\nedge proportion"
})

edge_group_order = [
    "Edge proportion",
    "Importance-weighted\nedge proportion"
]

edge_rho = edge_df.pivot(
    index="Feature",
    columns="Feature_group",
    values="rho"
)

edge_p = edge_df.pivot(
    index="Feature",
    columns="Feature_group",
    values="p_fdr"
)

order = (
    edge_rho
    .abs()
    .max(axis=1)
    .sort_values(ascending=False)
    .index
)

edge_rho = edge_rho.reindex(
    index=order,
    columns=edge_group_order
)

edge_p = edge_p.reindex(
    index=order,
    columns=edge_group_order
)


plot_corr_heatmap(
    rho_df=edge_rho.T,
    p_df=edge_p.T,
    title="Correlation between edge-level tissue features and model-derived risk",
    save_path=save_dir_edge,
    figsize=(8, 4),
    vmin=-0.35,
    vmax=0.35,
    show_rho=False
)